# Fase 3: Análisis de Tendencias - Clasificación de Auge y Declive

En este cuaderno diagnosticaremos el estado actual de las 20 tecnologías principales de Stack Overflow. 
Nuestro objetivo es determinar de manera matemática y estadística si una tecnología está **En Auge, Madurando o En Declive**.

In [1]:
import polars as pl
import numpy as np
import plotly.express as px
import sys
import os

# Importar funciones modulares desde src
sys.path.append(os.path.abspath('..'))
from src.metrics import crecimiento_anual, tendencia_lineal
from src.classifier import clasificar_tendencia

# Rutas de archivos
TOP20_PATH = '../data/datos_procesados/eda/top20.parquet'
DIM_QUESTIONS_PATH = '../data/datos_procesados/dim_questions.parquet'
FACT_TAGS_PATH = '../data/datos_procesados/fact_question_tags.parquet'
OUTPUT_PATH = '../data/datos_procesados/eda/clasificacion.parquet'


## 1. Carga de Datos y Cálculo de ViewCount Promedio (Nivel Global)

Para evitar el error matemático de "promediar promedios", calcularemos el promedio global de `ViewCount` por tecnología haciendo el JOIN directamente a nivel de pregunta, ANTES de hacer agregaciones mensuales.

In [2]:
# 1. Cargar Top 20 para filtrar
top20 = pl.read_parquet(TOP20_PATH)
top20_tags = top20['Tag'].to_list()

# 2. Cargar tablas con rename para normalizar 'Tags' a 'Tag'
questions_lazy = pl.scan_parquet(DIM_QUESTIONS_PATH)
tags_lazy = pl.scan_parquet(FACT_TAGS_PATH).rename({"Tags": "Tag"}) # <-- Aquí corregimos el ColumnNotFoundError

# Filtrar sólo los tags del Top 20
tags_top20 = tags_lazy.filter(pl.col('Tag').is_in(top20_tags))

# 3. Join a nivel global
global_join = questions_lazy.join(tags_top20, on='Id', how='inner').collect()

# 4. Cálculo del ViewCount Promedio GLOBAL por Tag
viewcount_promedio = global_join.group_by('Tag').agg(
    pl.col('ViewCount').mean().alias('ViewCount_Promedio')
)

print("Muestra del ViewCount Promedio calculado a nivel global:")
display(viewcount_promedio.head())


Muestra del ViewCount Promedio calculado a nivel global:


Tag,ViewCount_Promedio
str,f64
"""c++""",1121.632289
"""css""",1620.817452
"""android""",1821.030597
"""php""",1318.458252
"""arrays""",1275.698604


## 2. Resample Mensual, Crecimiento y Tendencia

Construimos nuestra serie temporal con frecuencia mensual contando la cantidad de preguntas. A esta serie le aplicaremos:
1. **Crecimiento % Anual**: Comparando el volumen del último año completo (2024) versus el año anterior (2023).
2. **Tendencia Lineal**: Calculando la pendiente (preguntas por mes) y el $R^2$ usando nuestra función basada en `numpy.polyfit`.

In [3]:
# Extraer Año y Mes para agrupamiento temporal
global_join = global_join.with_columns([
    pl.col('CreationDate').dt.year().alias('Year'),
    pl.col('CreationDate').dt.strftime('%Y-%m').alias('YearMonth')
])

# Agrupar por mes
monthly_series = global_join.group_by(['Tag', 'Year', 'YearMonth']).agg(
    pl.len().alias('Count')
).sort(['Tag', 'YearMonth'])

# Crear un índice numérico para el tiempo, necesario para la regresión lineal
monthly_series = monthly_series.with_columns(
    pl.col('YearMonth').rank('dense').over('Tag').alias('Mes_Indice')
)

# --- APLICAR MÉTODOS DE src/metrics.py ---

# 1. Crecimiento Porcentual (2022 vs 2024)
df_crecimiento = crecimiento_anual(monthly_series, count_col='Count', tag_col='Tag', year_col='Year')

# 2. Tendencia Lineal (Pendiente y R2)
df_tendencia = tendencia_lineal(monthly_series, time_col='Mes_Indice', count_col='Count', tag_col='Tag')

display(df_crecimiento.head())
display(df_tendencia.head())


Tag,Share_2022,Share_2023,Crecimiento_Pct
str,f64,f64,f64
"""python""",21.049626,19.81637,-5.8588
"""angular""",2.130515,2.580058,21.100209
"""c#""",5.840181,6.680559,14.389589
"""c++""",3.510044,3.334487,-5.00157
"""r""",4.343529,4.682977,7.815021


Tag,Pendiente,R2
str,f64,f64
"""c++""",-42.383556,0.813809
"""android""",-121.132362,0.955382
"""html""",-83.743138,0.921062
"""css""",-52.30845,0.904782
"""python-3.x""",6.832494,0.019745


## 3. Clasificador por Reglas

Integrando nuestra función de `src.classifier`, clasificaremos cada tecnología basada en su crecimiento:
- **En Auge**: $\geq 20\%$
- **Madurando**: Entre $-5\%$ y $20\%$
- **En Declive**: $< -5\%$

In [4]:
# Consolidar todas las métricas en un solo DataFrame
df_final = (
    top20.rename({"Count": "Volumen_Total"})
    .select(['Tag', 'Volumen_Total'])
    .join(df_crecimiento, on='Tag', how='left')
    .join(df_tendencia, on='Tag', how='left')
    .join(viewcount_promedio, on='Tag', how='left')
)

# Clasificar
df_final = df_final.with_columns(
    clasificar_tendencia('Crecimiento_Pct', 'Categoria_Tendencia')
)


## 4. Consolidación y Visualizaciones Requeridas

A continuación, visualizamos el posicionamiento estratégico. 

**Gráfico de Burbujas**: Nos permite ver 3 dimensiones en simultáneo: Volumen Absoluto (X), Crecimiento (Y) y Popularidad en Lecturas (Tamaño).

In [5]:
df_pd = df_final.to_pandas()

fig1 = px.scatter(
    df_pd,
    x='Volumen_Total',
    y='Crecimiento_Pct',
    size='ViewCount_Promedio',
    color='Categoria_Tendencia',
    text='Tag',
    title='Bubble Chart: Posicionamiento Estratégico de Tecnologías Top 20',
    labels={
        'Volumen_Total': 'Volumen Total (Absoluto)', 
        'Crecimiento_Pct': 'Crecimiento Anual % (2024 vs 2023)',
        'ViewCount_Promedio': 'Vistas Promedio'
    },
    color_discrete_map={'En Auge': '#2ca02c', 'Madurando': '#ff7f0e', 'En Declive': '#d62728'},
    size_max=60
)

# Añadir líneas de referencia para las reglas del clasificador
fig1.add_hline(y=20, line_dash="dash", line_color="green", opacity=0.5)
fig1.add_hline(y=-5, line_dash="dash", line_color="red", opacity=0.5)

fig1.update_traces(textposition='top center')
fig1.update_layout(template='plotly_white', height=700, margin=dict(l=40, r=40, t=60, b=40))
fig1.show()


### Crecimiento % Ordenado

In [6]:
fig2 = px.bar(
    df_pd.sort_values('Crecimiento_Pct', ascending=False),
    x='Tag',
    y='Crecimiento_Pct',
    color='Categoria_Tendencia',
    title='Crecimiento % Anual por Tecnología (2022 vs 2023)',
    labels={'Crecimiento_Pct': 'Crecimiento Anual %', 'Tag': 'Tecnología'},
    color_discrete_map={'En Auge': '#2ca02c', 'Madurando': '#ff7f0e', 'En Declive': '#d62728'},
    text_auto='.2f'
)
fig2.update_layout(template='plotly_white', height=500, xaxis_tickangle=-45)
fig2.update_traces(textposition='outside')
fig2.show()


### Tabla Resumen Completa

Aquí imprimimos el DataFrame consolidado estilizado para poder inspeccionar las cifras exactas de `Pendiente`, $R^2$ y otras métricas.

In [9]:
# Mostrar tabla con formato
# Usaremos estilo de Pandas para resaltar colores dependiendo de la categoría
def color_categoria(val):
    color = '#2ca02c' if val == 'En Auge' else '#ff7f0e' if val == 'Madurando' else '#d62728'
    return f'color: {color}; font-weight: bold'

styled_df = (
    df_pd.sort_values('Crecimiento_Pct', ascending=False)
    .style
    .format({
        'Crecimiento_Pct': '{:.2f}%',
        'Pendiente': '{:.4f}',
        'R2': '{:.4f}',
        'ViewCount_Promedio': '{:.0f}'
    })
    .map(color_categoria, subset=['Categoria_Tendencia'])
    .background_gradient(cmap='Blues', subset=['Volumen_Total'])
)

display(styled_df)


,Tag,Volumen_Total,Share_2022,Share_2023,Crecimiento_Pct,Pendiente,R2,ViewCount_Promedio,Categoria_Tendencia
14,ios,407440,1.461375,1.868922,27.89%,-74.0368,0.9047,1691,En Auge
4,android,858772,3.969203,5.039078,26.95%,-121.1324,0.9554,1821,En Auge
18,angular,303776,2.130515,2.580058,21.10%,-3.4556,0.0052,3412,En Auge
3,c#,909857,5.840181,6.680559,14.39%,-98.7541,0.9577,1448,Madurando
2,java,1186673,6.412916,7.064100,10.15%,-145.4063,0.9586,1734,Madurando
17,swift,317837,1.672476,1.832946,9.59%,-37.5799,0.8911,1657,Madurando
11,r,431535,4.343529,4.682977,7.82%,-6.0230,0.0539,1076,Madurando
13,node.js,411057,3.808124,4.068793,6.85%,-8.7988,0.1127,2313,Madurando
10,reactjs,474660,7.414293,7.826608,5.56%,49.7599,0.5202,2701,Madurando
7,css,530506,3.754369,3.933731,4.78%,-52.3085,0.9048,1621,Madurando


## 5. Exportación

Guardamos nuestro análisis en formato parquet. Este dataset (`clasificacion.parquet`) será un insumo crucial para las métricas consolidadas de la **Fase 4** (Dashboard y API).

In [8]:
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
df_final.write_parquet(OUTPUT_PATH)
print(f"Resultados consolidados guardados en: {OUTPUT_PATH}")


Resultados consolidados guardados en: ../data/datos_procesados/eda/clasificacion.parquet
